# Replacement-Dynamics Adoption Model by FSA

This notebook extends the province-level replacement-dynamics model to the **FSA level**.

The idea is the same as in the aggregate notebook, but applied to each postal area separately:

1. Estimate how the **sales share** of each vehicle type changes over time using historical SAAQ entries.
2. Replace `inf`, `-inf`, and `NaN` adoption rates with `0`, so zero-base years do not create explosive growth.
3. Keep the **fleet size smooth**, using the observed average yearly net fleet change for each FSA instead of exponential growth.
4. Add future vehicles using the projected sales share.
5. Remove vehicles proportionally from the existing fleet, so the fleet composition changes gradually through replacement.

This gives a more realistic dynamic than projecting fleet share directly, because it separates:

- **sales dynamics**: which technologies are being added,
- **replacement dynamics**: how fast the existing fleet can actually change.

The notebook has two goals:

- inspect the model carefully for **one selected FSA**,
- generate a consistent **summary for all FSAs**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

PROJECT_DIR = Path('/Users/natomanzolli/Documents/GitHub/MATSim-agent-vehicle-assignment/adoption prediction model')
CACHE_DIR = PROJECT_DIR / '.cache'

FLEET_COUNTS_CACHE = CACHE_DIR / 'saq_vehicle_counts.pkl'
ENTRY_COUNTS_CACHE = CACHE_DIR / 'saq_entry_counts.pkl'

fleet_counts = pd.read_pickle(FLEET_COUNTS_CACHE)
entry_counts = pd.read_pickle(ENTRY_COUNTS_CACHE)

fleet_counts.head()

## Settings

This notebook lets you control the FSA analysis with a few explicit parameters.

Important choices:

- `ENTRY_FLAG`: use `Entrant` or `Neuf` as the proxy for new sales entering the fleet.
- `TARGET_FSA`: the FSA shown in the detailed diagnostic plots.
- `FORECAST_END_YEAR`: final projection year.
- `USE_RECENT_YEARS_ONLY`: if `True`, the averages use only recent years.
- `ADOPTION_RATE_CLIP`: caps extreme FSA-level adoption rates to avoid noisy small-sample jumps.
- `MIN_AVG_ENTRIES_FOR_FLAG`: threshold used only for diagnostics, to flag sparse FSAs.

In [ ]:
ENTRY_FLAG = 'Entrant'
TARGET_FSA = 'H2X'
FORECAST_END_YEAR = 2035
USE_RECENT_YEARS_ONLY = False
RECENT_YEAR_START = 2017
ADOPTION_RATE_CLIP = (-0.75, 1.50)
MIN_AVG_ENTRIES_FOR_FLAG = 25

vehicle_order = ['electric', 'hev_sedan', 'hev_suv', 'ice_sedan', 'ice_suv', 'ice_van/pickup']

## Step 1: Prepare Historical FSA Tables

We convert the cached long tables into FSA-year pivots for:

- fleet counts by vehicle type,
- yearly entry counts by vehicle type,
- total fleet size,
- total yearly entries.

This gives a clean base for the replacement model.

In [ ]:
fleet_pivot = (
    fleet_counts.pivot_table(
        index=['fsa', 'AnneeSAAQ'],
        columns='vehicle_type',
        values='count',
        aggfunc='sum',
        fill_value=0,
    )
    .reindex(columns=vehicle_order, fill_value=0)
    .sort_index()
)
fleet_pivot['total_vehicles'] = fleet_pivot.sum(axis=1)

entry_pivot = (
    entry_counts.pivot_table(
        index=['fsa', 'AnneeSAAQ'],
        columns='vehicle_type',
        values=ENTRY_FLAG,
        aggfunc='sum',
        fill_value=0,
    )
    .reindex(columns=vehicle_order, fill_value=0)
    .sort_index()
)
entry_pivot['total_entries'] = entry_pivot.sum(axis=1)

pd.DataFrame({
    'fleet_rows': [len(fleet_pivot)],
    'entry_rows': [len(entry_pivot)],
    'n_fsas': [fleet_pivot.index.get_level_values('fsa').nunique()],
    'year_min': [fleet_pivot.index.get_level_values('AnneeSAAQ').min()],
    'year_max': [fleet_pivot.index.get_level_values('AnneeSAAQ').max()],
})

## Step 2: Define the FSA Replacement Model

This helper function applies the full replacement logic to one FSA:

1. Historical fleet counts.
2. Historical entry shares.
3. Historical adoption rates of sales share.
4. Historical average net fleet change.
5. Historical average entries.
6. Future sales-share projection.
7. Future fleet projection with additions and removals.

A few safeguards are included:

- all missing and infinite adoption rates become `0`,
- adoption rates are clipped to a configurable interval,
- if a share vector collapses to zero, the previous share is reused,
- yearly exits are constrained to stay nonnegative,
- vehicle counts are clipped at zero.

In [ ]:
def run_replacement_model_for_fsa(
    fsa,
    fleet_pivot,
    entry_pivot,
    vehicle_cols,
    forecast_end_year=2035,
    use_recent_years_only=False,
    recent_year_start=2017,
    adoption_rate_clip=(-0.75, 1.50),
):
    try:
        fleet_hist = fleet_pivot.xs(fsa, level='fsa').copy()
        entry_hist = entry_pivot.xs(fsa, level='fsa').copy()
    except KeyError:
        raise KeyError(f'FSA {fsa} not found in cached data.')

    fleet_hist = fleet_hist.sort_index()
    entry_hist = entry_hist.sort_index()

    all_years = sorted(set(fleet_hist.index).union(entry_hist.index))
    fleet_hist = fleet_hist.reindex(all_years, fill_value=0)
    entry_hist = entry_hist.reindex(all_years, fill_value=0)

    fleet_hist['total_vehicles'] = fleet_hist[vehicle_cols].sum(axis=1)
    entry_hist['total_entries'] = entry_hist[vehicle_cols].sum(axis=1)

    entry_share_hist = entry_hist[vehicle_cols].div(entry_hist['total_entries'].replace(0, np.nan), axis=0)
    entry_share_hist = entry_share_hist.fillna(0)

    adoption_rate = entry_share_hist.pct_change()
    adoption_rate = adoption_rate.replace([np.inf, -np.inf], 0).fillna(0)
    if adoption_rate_clip is not None:
        adoption_rate = adoption_rate.clip(lower=adoption_rate_clip[0], upper=adoption_rate_clip[1])

    adoption_source = adoption_rate.copy()
    if use_recent_years_only:
        adoption_source = adoption_source[adoption_source.index >= recent_year_start]
    avg_adoption_rate = adoption_source.mean().reindex(vehicle_cols).fillna(0)

    fleet_change = fleet_hist[['total_vehicles']].diff().rename(columns={'total_vehicles': 'fleet_net_change'})
    fleet_change['fleet_growth_rate'] = fleet_hist['total_vehicles'].pct_change()
    fleet_change_source = fleet_change.dropna().copy()
    if use_recent_years_only:
        fleet_change_source = fleet_change_source[fleet_change_source.index >= recent_year_start]

    avg_net_change = float(fleet_change_source['fleet_net_change'].mean()) if len(fleet_change_source) else 0.0

    entry_source = entry_hist[['total_entries']].copy()
    if use_recent_years_only:
        entry_source = entry_source[entry_source.index >= recent_year_start]
    avg_entries_per_year = float(entry_source['total_entries'].mean()) if len(entry_source) else 0.0

    last_observed_year = int(max(all_years))
    future_years = list(range(last_observed_year + 1, forecast_end_year + 1))

    sales_share_future = entry_share_hist.copy()
    if last_observed_year in sales_share_future.index:
        current_sales_share = sales_share_future.loc[last_observed_year, vehicle_cols].astype(float).copy()
    else:
        current_sales_share = pd.Series(1 / len(vehicle_cols), index=vehicle_cols, dtype=float)

    if current_sales_share.sum() <= 0:
        current_sales_share = pd.Series(1 / len(vehicle_cols), index=vehicle_cols, dtype=float)
    else:
        current_sales_share = current_sales_share / current_sales_share.sum()

    for year in future_years:
        next_share = current_sales_share * (1 + avg_adoption_rate)
        next_share = next_share.clip(lower=0)
        if next_share.sum() > 0:
            next_share = next_share / next_share.sum()
        else:
            next_share = current_sales_share.copy()
        sales_share_future.loc[year, vehicle_cols] = next_share
        current_sales_share = next_share.copy()

    sales_share_future = sales_share_future.sort_index()

    projected_counts = fleet_hist.copy()
    current_counts = projected_counts.loc[last_observed_year, vehicle_cols].astype(float).copy()
    projection_rows = []

    for year in future_years:
        entries = float(avg_entries_per_year)
        exits = max(entries - avg_net_change, 0.0)

        current_total = current_counts.sum()
        if current_total > 0:
            current_fleet_share = current_counts / current_total
        else:
            current_fleet_share = pd.Series(1 / len(vehicle_cols), index=vehicle_cols, dtype=float)

        forecast_sales_share = sales_share_future.loc[year, vehicle_cols].astype(float)
        additions = entries * forecast_sales_share
        removals = exits * current_fleet_share

        next_counts = (current_counts + additions - removals).clip(lower=0)
        next_total = float(next_counts.sum())

        projected_counts.loc[year, vehicle_cols] = next_counts
        projected_counts.loc[year, 'total_vehicles'] = next_total

        projection_rows.append({
            'fsa': fsa,
            'year': year,
            'entries': entries,
            'exits': exits,
            'net_change': entries - exits,
            'projected_total_vehicles': next_total,
        })

        current_counts = next_counts.copy()

    projected_counts = projected_counts.sort_index()
    projected_market_share = projected_counts[vehicle_cols].div(projected_counts[vehicle_cols].sum(axis=1), axis=0).fillna(0)

    diagnostics = {
        'fsa': fsa,
        'n_hist_years': int(len(all_years)),
        'year_min': int(min(all_years)),
        'year_max': int(max(all_years)),
        'avg_entries_per_year': avg_entries_per_year,
        'avg_net_change': avg_net_change,
        'last_observed_total': float(fleet_hist.loc[last_observed_year, 'total_vehicles']),
        'projected_2035_total': float(projected_counts.loc[forecast_end_year, 'total_vehicles']) if forecast_end_year in projected_counts.index else np.nan,
    }

    return {
        'fleet_hist': fleet_hist,
        'entry_hist': entry_hist,
        'entry_share_hist': entry_share_hist,
        'adoption_rate': adoption_rate,
        'avg_adoption_rate': avg_adoption_rate,
        'fleet_change': fleet_change,
        'avg_net_change': avg_net_change,
        'avg_entries_per_year': avg_entries_per_year,
        'sales_share_future': sales_share_future,
        'projected_counts': projected_counts,
        'projected_market_share': projected_market_share,
        'projection_summary': pd.DataFrame(projection_rows),
        'diagnostics': diagnostics,
    }

## Step 3: Run the Model for One Selected FSA

This section is the detailed walkthrough. It helps you inspect one postal area and verify whether the assumptions make sense locally.

In [ ]:
selected = run_replacement_model_for_fsa(
    TARGET_FSA,
    fleet_pivot=fleet_pivot,
    entry_pivot=entry_pivot,
    vehicle_cols=vehicle_order,
    forecast_end_year=FORECAST_END_YEAR,
    use_recent_years_only=USE_RECENT_YEARS_ONLY,
    recent_year_start=RECENT_YEAR_START,
    adoption_rate_clip=ADOPTION_RATE_CLIP,
)

pd.DataFrame([selected['diagnostics']])

## Step 4: Inspect Historical Sales Shares and Adoption Rates

At the FSA level, sales-share series can be noisy, especially for small markets. That is why the notebook shows both:

- the observed sales share by vehicle type,
- the historical adoption-rate table used to project the future.

If an FSA is very small, these rates may still be volatile, even after replacing infinities with zero and clipping extremes.

In [ ]:
selected['entry_share_hist']

In [ ]:
selected['adoption_rate']

## Step 5: Detailed FSA Plots

These figures answer the key modeling questions for the selected FSA:

- Is the fleet size staying smooth?
- Are the projected sales shares moving in a plausible direction?
- Does the fleet market share change more slowly than the sales mix?
- Are entry and exit volumes consistent with the stable-fleet assumption?

In [ ]:
fig, ax = plt.subplots()
selected['projected_counts']['total_vehicles'].plot(ax=ax, marker='o', label='Projected total fleet')
selected['fleet_hist']['total_vehicles'].plot(ax=ax, marker='o', linewidth=2.5, label='Observed total fleet')
ax.set_title(f'{TARGET_FSA}: Total Fleet Size')
ax.set_xlabel('Year')
ax.set_ylabel('Vehicles')
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
selected['sales_share_future'][vehicle_order].plot(ax=ax, marker='o')
ax.set_title(f'{TARGET_FSA}: Historical and Future Sales Share')
ax.set_xlabel('Year')
ax.set_ylabel('Sales share')
ax.legend(title='Vehicle type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
selected['projected_market_share'][vehicle_order].plot(ax=ax, marker='o')
ax.set_title(f'{TARGET_FSA}: Fleet Market Share')
ax.set_xlabel('Year')
ax.set_ylabel('Fleet share')
ax.legend(title='Vehicle type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
selected['projected_counts'][vehicle_order].plot(ax=ax, marker='o')
ax.set_title(f'{TARGET_FSA}: Vehicle Counts by Type')
ax.set_xlabel('Year')
ax.set_ylabel('Vehicles')
ax.legend(title='Vehicle type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
selected['entry_hist']['total_entries'].plot(ax=ax, marker='o', label='Observed entries')
if not selected['projection_summary'].empty:
    selected['projection_summary'].set_index('year')['entries'].plot(ax=ax, linestyle='--', marker='o', label='Forecast entries')
    selected['projection_summary'].set_index('year')['exits'].plot(ax=ax, linestyle='--', marker='o', label='Forecast exits')
ax.set_title(f'{TARGET_FSA}: Entries and Implied Exits')
ax.set_xlabel('Year')
ax.set_ylabel('Vehicles')
ax.legend()
plt.show()

## Step 6: Run the Model for All FSAs

Now we apply the same logic to every FSA in the cache.

This produces a compact set of summary tables that help identify:

- FSAs with strong EV sales-share growth,
- FSAs with shrinking or expanding fleet size,
- sparse FSAs where interpretation should be cautious.

In [ ]:
all_results = {}
summary_rows = []
forecast_rows = []

for fsa in sorted(fleet_pivot.index.get_level_values('fsa').unique()):
    result = run_replacement_model_for_fsa(
        fsa,
        fleet_pivot=fleet_pivot,
        entry_pivot=entry_pivot,
        vehicle_cols=vehicle_order,
        forecast_end_year=FORECAST_END_YEAR,
        use_recent_years_only=USE_RECENT_YEARS_ONLY,
        recent_year_start=RECENT_YEAR_START,
        adoption_rate_clip=ADOPTION_RATE_CLIP,
    )
    all_results[fsa] = result

    diag = result['diagnostics'].copy()
    diag['sparse_fsa_flag'] = diag['avg_entries_per_year'] < MIN_AVG_ENTRIES_FOR_FLAG

    market_share = result['projected_market_share'].copy()
    for year in [2021, FORECAST_END_YEAR]:
        if year in market_share.index:
            diag[f'electric_fleet_share_{year}'] = float(market_share.loc[year, 'electric'])
            diag[f'ice_sedan_fleet_share_{year}'] = float(market_share.loc[year, 'ice_sedan'])
            diag[f'ice_suv_fleet_share_{year}'] = float(market_share.loc[year, 'ice_suv'])

    if 2021 in market_share.index and FORECAST_END_YEAR in market_share.index:
        diag['electric_share_change_2021_to_2035'] = float(
            market_share.loc[FORECAST_END_YEAR, 'electric'] - market_share.loc[2021, 'electric']
        )

    summary_rows.append(diag)

    market_share_reset = market_share.reset_index().rename(columns={'index': 'year'})
    market_share_reset.insert(0, 'fsa', fsa)
    forecast_rows.append(market_share_reset)

fsa_summary = pd.DataFrame(summary_rows).sort_values('electric_share_change_2021_to_2035', ascending=False)
fsa_forecast_market_share = pd.concat(forecast_rows, ignore_index=True)

fsa_summary.head(10)

## Step 7: Province-Wide FSA Diagnostics

These tables help you read the FSA results more carefully.

Recommended interpretation:

- high EV growth with high average yearly entries is more credible,
- high EV growth with very low entries should be treated cautiously,
- projected total fleet changes should remain smooth under this framework.

In [ ]:
fsa_summary[['fsa', 'avg_entries_per_year', 'avg_net_change', 'last_observed_total', 'projected_2035_total', 'sparse_fsa_flag', 'electric_fleet_share_2021', 'electric_fleet_share_2035', 'electric_share_change_2021_to_2035']].head(20)

In [ ]:
fsa_summary[['avg_entries_per_year', 'avg_net_change', 'electric_fleet_share_2021', 'electric_fleet_share_2035', 'electric_share_change_2021_to_2035']].describe().T

In [ ]:
fsa_summary.sort_values(['sparse_fsa_flag', 'electric_share_change_2021_to_2035'], ascending=[True, False])[['fsa', 'avg_entries_per_year', 'sparse_fsa_flag', 'electric_fleet_share_2021', 'electric_fleet_share_2035', 'electric_share_change_2021_to_2035']].head(25)

## Step 8: FSA-Level Summary Plots

These are not meant to replace the individual FSA analysis. They are portfolio-level diagnostics that show how the model behaves across postal areas.

In [ ]:
fig, ax = plt.subplots()
sns.histplot(fsa_summary['electric_fleet_share_2035'], bins=30, ax=ax)
ax.set_title('Distribution of Projected 2035 EV Fleet Share Across FSAs')
ax.set_xlabel('Projected EV fleet share in 2035')
ax.set_ylabel('Number of FSAs')
plt.show()

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(
    data=fsa_summary,
    x='avg_entries_per_year',
    y='electric_share_change_2021_to_2035',
    hue='sparse_fsa_flag',
    ax=ax,
)
ax.set_title('EV Share Change vs Average Entries per Year')
ax.set_xlabel('Average entries per year')
ax.set_ylabel('EV fleet share change (2021 to 2035)')
plt.show()

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(
    data=fsa_summary,
    x='last_observed_total',
    y='projected_2035_total',
    hue='sparse_fsa_flag',
    ax=ax,
)
ax.set_title('Observed Last Fleet Size vs Projected 2035 Fleet Size')
ax.set_xlabel('Observed fleet size in last historical year')
ax.set_ylabel('Projected fleet size in 2035')
plt.show()

## Optional: Compare Several FSAs at Once

This cell is useful if you want to compare a few postal areas directly. Edit the list and rerun.

In [ ]:
COMPARE_FSAS = ['H2X', 'G1K', 'J7V', 'H7N']

comparison_rows = []
for fsa in COMPARE_FSAS:
    if fsa in all_results:
        share = all_results[fsa]['projected_market_share'].copy()
        for year in [2021, FORECAST_END_YEAR]:
            if year in share.index:
                comparison_rows.append({
                    'fsa': fsa,
                    'year': year,
                    'electric': float(share.loc[year, 'electric']),
                    'hev_sedan': float(share.loc[year, 'hev_sedan']),
                    'hev_suv': float(share.loc[year, 'hev_suv']),
                    'ice_sedan': float(share.loc[year, 'ice_sedan']),
                    'ice_suv': float(share.loc[year, 'ice_suv']),
                    'ice_van/pickup': float(share.loc[year, 'ice_van/pickup']),
                })
comparison_df = pd.DataFrame(comparison_rows)
comparison_df

In [ ]:
if not comparison_df.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    for fsa in COMPARE_FSAS:
        if fsa in all_results:
            series = all_results[fsa]['projected_market_share']['electric']
            ax.plot(series.index, series.values, marker='o', label=fsa)
    ax.set_title('Projected EV Fleet Share for Selected FSAs')
    ax.set_xlabel('Year')
    ax.set_ylabel('EV fleet share')
    ax.legend(title='FSA', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## Optional: Save Outputs

The final cell exports the two most useful tables:

- `fsa_replacement_dynamics_summary.csv`
- `fsa_replacement_dynamics_market_share.csv`

This gives you a reusable summary without rerunning the notebook plots.

In [ ]:
output_dir = PROJECT_DIR / 'validation_outputs' / 'replacement_dynamics_adoption_model_fsa'
output_dir.mkdir(parents=True, exist_ok=True)

fsa_summary.to_csv(output_dir / 'fsa_replacement_dynamics_summary.csv', index=False)
fsa_forecast_market_share.to_csv(output_dir / 'fsa_replacement_dynamics_market_share.csv', index=False)

output_dir